In [ ]:
!pip install python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 2.5 MB/s eta 0:00:00


In [ ]:
!pip install -q tiktoken==0.5.2 langchain_community==0.0.11 langchain==0.1.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.0/798.0 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.9/302.9 kB 17.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.9/302.9 kB 18.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 21.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.0/303.0 kB 23.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.9/302.9 kB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.3/299.3 kB 20.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━

In [ ]:
from google.colab import userdata
import numpy as np
import os
import io
import tempfile
import tiktoken
from docx import Document
from scipy.spatial.distance import cdist
import json
import re
import requests
from google.colab import userdata
from langchain.text_splitter import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter, CharacterTextSplitter
from langchain.docstore.document import Document as Doc
import pickle

In [ ]:
ID_FOLDER = userdata.get('ID_FOLDER')
OAuth_token = userdata.get('OAuth_token')

In [ ]:
# URL для получения токена
URL = "https://iam.api.cloud.yandex.net/iam/v1/tokens"

# Получение IAM-токена (с помощью request)
import requests

headers = {"Content-Type": "application/json"}

data = {
    "yandexPassportOauthToken": OAuth_token
}

response = requests.post(URL, headers=headers, json=data)

IAM_TOKEN = response.json()["iamToken"]
expiresAt = response.json()["expiresAt"]

print(f'Ваш токен действителен до: {expiresAt}')

Ваш токен действителен до: 2024-05-15T18:41:07.495856603Z


In [ ]:
#Готовая база для Markdown: v3.0: https://drive.google.com/file/d/1e4U1IoxumnkW0ttZ-n3CDnDMKX4Qvco-/view?usp=sharing
id_doc_markdown = '1e4U1IoxumnkW0ttZ-n3CDnDMKX4Qvco-'

In [ ]:
# функция для загрузки документа по doc_id из гугл драйв в текстовом формате
def load_doc_id_text(file_id):
    # Download the document as plain text
    response = requests.get(f'https://drive.google.com/uc?export=download&id={file_id}')
    response.raise_for_status()
    text = response.text

    return text

In [ ]:
data_from_markdown = load_doc_id_text(id_doc_markdown)

In [ ]:
def num_tokens_from_string(string: str, encoding_name: str) -> int:
      """Возвращает количество токенов в строке"""
      encoding = tiktoken.get_encoding(encoding_name)
      num_tokens = len(encoding.encode(string))
      return num_tokens

def split_text(text, max_count):
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
        ("####", "Header 4"),
        ("#####", "Header 5"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
    fragments = markdown_splitter.split_text(text)

    # Подсчет токенов для каждого фрагмента
    fragment_token_counts = [num_tokens_from_string(fragment.page_content, "cl100k_base") for fragment in fragments]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_count,
        chunk_overlap=0,
        length_function=lambda x: num_tokens_from_string(x, "cl100k_base")
    )

    source_chunks = [
        Doc(page_content=chunk, metadata=fragment.metadata)
        for fragment in fragments
        for chunk in splitter.split_text(fragment.page_content)
    ]

    # Подсчет токенов для каждого source_chunk
    source_chunk_token_counts = [num_tokens_from_string(chunk.page_content, "cl100k_base") for chunk in source_chunks]

    return source_chunks, fragments

In [ ]:
source_chunks, fragments = split_text(data_from_markdown, 1000)
print("Общее количество чанков: ",len(source_chunks))
print("Первый чанк ", source_chunks[0])
print("Metadata Первого чанка", source_chunks[0].metadata)
print("Крайний чанк ", source_chunks[len(source_chunks)-1])
print("Metadata крайнего чанка", source_chunks[len(source_chunks)-1].metadata)


Общее количество чанков:  1266
Первый чанк  page_content='Документ предоставлен КонсультантПлюс'
Metadata Первого чанка {}
Крайний чанк  page_content='5. На оборотной стороне свидетельств о поверке при оформлении их на бумажном носителе или в свидетельствах о поверке, оформляемых в виде электронного документа, по заявлению владельцев средств измерений или лиц, представивших средства измерений на поверку, или по согласованию с ними может указываться дополнительная информация, относящаяся к средствам измерений, месту их установки, особенностям поверки, включая сведения о пломбах, предотвращающих доступ к местам настройки (регулировки) средств измерений, принадлежности средств измерений (сведения о владельцах средств измерений), а также информация о прилагаемых к свидетельству о поверке документах (при невозможности размещения информации на оборотной стороне свидетельства о поверке при оформлении его на бумажном носителе).' metadata={'Header 1': 'ТРЕБОВАНИЯ К СОДЕРЖАНИЮ СВИДЕТЕЛЬСТВА О ПО

In [ ]:
separator = ","
result = separator.join([f"{k}. {v}" for k, v in source_chunks[len(source_chunks)-1].metadata.items()])
print (result)

Header 1. ТРЕБОВАНИЯ К СОДЕРЖАНИЮ СВИДЕТЕЛЬСТВА О ПОВЕРКЕ,Header 2. пункт 5.


In [ ]:
#Переведем все метадата чанков в вид, который необходим для Embedding YandexGPT
doc_metadata = []
separator = ","
for i in range(len(source_chunks)):
    result = separator.join([f"{k}. {v}" for k, v in source_chunks[i].metadata.items()])
    text = f'"""{result}""",'
    doc_metadata.append(text)

In [ ]:
doc_metadata

In [ ]:
print(type(source_chunks[len(source_chunks)-1].metadata))
txt_str = f'{json.dumps(source_chunks[len(source_chunks)-1].metadata, ensure_ascii=False)}'
print (type(txt_str), txt_str)

<class 'dict'>
<class 'str'> {"Header 1": "ТРЕБОВАНИЯ К СОДЕРЖАНИЮ СВИДЕТЕЛЬСТВА О ПОВЕРКЕ"}


In [ ]:
#Переведем все чанки в вид, который необходим для Embedding YandexGPT
doc_texts = []
for i in range(len(source_chunks)):
    text = f'"""{source_chunks[i].page_content}""",'
    doc_texts.append(text)

In [ ]:
#Сохраним подготовленную базу в файл
with open('doc_texts_03.pkl', 'wb') as file:
    pickle.dump(doc_texts, file)

In [ ]:
#Выгрузим подготовленную базу для embedding из файла
with open ('doc_texts_03.pkl', 'rb') as file :
    doc_texts = pickle.load(file)

In [ ]:
print (type(doc_texts), len(doc_texts))

<class 'list'> 1266


In [ ]:
#Embedding YandexGPT
doc_uri = f"emb://{ID_FOLDER}/text-search-doc/latest"
query_uri = f"emb://{ID_FOLDER}/text-search-query/latest"

embed_url = "https://llm.api.cloud.yandex.net:443/foundationModels/v1/textEmbedding"
headers = {"Content-Type": "application/json", "Authorization": f"Bearer {IAM_TOKEN}", "x-folder-id": f"{ID_FOLDER}"}



def get_embedding(text: str, text_type: str = "doc") -> np.array:
    query_data = {
        "modelUri": doc_uri if text_type == "doc" else query_uri,
        "text": text,
    }

    return np.array(
        requests.post(embed_url, json=query_data, headers=headers).json()["embedding"]
    )

In [ ]:
#Embedding базы знаний
docs_embedding = [get_embedding(doc_text) for doc_text in doc_texts]
#docs_embedding

In [ ]:
#Сохраним Embedding базы знаний в файл
np.save('docs_embedding_03.npy', docs_embedding)

In [ ]:
#загрузка массива embeddings
docs_embedding = np.load('docs_embedding_03.npy')

In [ ]:
print(type(docs_embedding), len(docs_embedding))

<class 'numpy.ndarray'> 1266


In [ ]:
# most similar doc text
#print(np.argmax(sim), doc_texts[np.argmax(sim)])

In [ ]:
#функция вывода k релевантных чанков по косинусному расстоянию
def get_k_max_indices(arr, k):
    indices = np.argpartition(arr, -k)[-k:]
    return indices

In [ ]:
system = "Ты юридический консультант по метрологическому обеспечению, ответь подробно на вопрос клиента на основании отрывков представленных ответов. При ответе ссылайся на статьи, главы и пункты нормативной документации. Не придумывай ничего от себя. Не отвечай на вопросы, не касающиеся метрологического обеспечения."

In [ ]:
#Функция обращения к модели YandexGPT Pro
def get_gpt_ya_response (ID_FOLDER, question):
    prompt = {
        "modelUri": f"gpt://{ID_FOLDER}/yandexgpt", # yandexgpt - модель YandexGPT pro
        "completionOptions": {
            "stream": False,
            "temperature": 0.0,
            "maxTokens": 2000
        },
            "messages": [
                {
                "role": "system",
                "text": f"{system}"
                },
                {
                "role": "user",
                "text": f"{question}"
                }
            ]
    }


    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {IAM_TOKEN}"
    }

    response = requests.post(url, headers=headers, json=prompt)
    json_data = json.loads(response.text)
    answer = json_data['result']['alternatives'][0]['message']['text']
    return answer

In [ ]:
#Функция получения релевантных чанков и метаданных и подготовка вопроса в LLM:
def chunks_metadata_question(query_text):
    #получем embedding вопроса
    query_embedding = get_embedding(query_text, text_type="query")
    # Вычисляем косинусное расстояние
    dist = cdist(query_embedding[None, :], docs_embedding, metric="cosine")
    # Вычисляем косинусное сходство
    sim = 1 - dist
    k = 3 #количество релевантных чанков
    indices = get_k_max_indices(sim, k)
    #находим индексы k релевантных чанков
    ind_k = indices[0][-k:]
    #выведем косинусное расстояние для найденных чанков
    for ind in ind_k: print(sim[0, ind])
    #Подтягиваем релевантные чанки из базы данных
    docs = []
    doc_meta = []
    for ind in ind_k:
        docs.append(doc_texts[ind])
        doc_meta.append(doc_metadata[ind])
    message_content = re.sub(r'\n{2}', ' ', '\n '.join([f'\nОтрывок документа №{i+1}\n=====================' + doc_meta[i] + ', ' + doc + '\n' for i, doc in enumerate(docs)]))
    print (message_content)
    question = f"Вопрос клиента: \n{query_text}\nОтветь на вопрос клиента. Не упоминай документ с информацией для ответа клиенту в ответе. Документ с информацией для ответа клиенту: {message_content}"
    return question


In [ ]:
#Доработка обращения к YandexGPT
query_text = 'Какой нормативный правовой акт устанавливает перечень измерений, реализуемых в сфере государственного регулирования обеспечения единства измерений при выполнении поручений суда, органов прокуратуры, государственных органов исполнительной власти, в том числе обязательные метрологические требования к ним.'
gpt_answer = get_gpt_ya_response(ID_FOLDER, chunks_metadata_question(query_text))
print ("Ответ YaGPT:\n", "=================\n", gpt_answer)

0.5032664369360146
0.5060329891453981
0.518690651889786

Отрывок документа №1
====================="""Header 1. ПРАВИТЕЛЬСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ ПОСТАНОВЛЕНИЕ от 20 апреля 2010 г. N 250 О ПЕРЕЧНЕ СРЕДСТВ ИЗМЕРЕНИЙ, ПОВЕРКА КОТОРЫХ ОСУЩЕСТВЛЯЕТСЯ ТОЛЬКО АККРЕДИТОВАННЫМИ В УСТАНОВЛЕННОМ ПОРЯДКЕ В ОБЛАСТИ ОБЕСПЕЧЕНИЯ ЕДИНСТВА ИЗМЕРЕНИЙ ГОСУДАРСТВЕННЫМИ РЕГИОНАЛЬНЫМИ ЦЕНТРАМИ МЕТРОЛОГИИ""",, """В соответствии со статьей 13 Федерального закона "Об обеспечении единства измерений" Правительство Российской Федерации постановляет:""",  
Отрывок документа №2
====================="""Header 1. ПРАВИТЕЛЬСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ ПОСТАНОВЛЕНИЕ от 16 ноября 2020 г. N 1847 ОБ УТВЕРЖДЕНИИ ПЕРЕЧНЯ ИЗМЕРЕНИЙ, ОТНОСЯЩИХСЯ К СФЕРЕ ГОСУДАРСТВЕННОГО РЕГУЛИРОВАНИЯ ОБЕСПЕЧЕНИЯ ЕДИНСТВА ИЗМЕРЕНИЙ""",, """В соответствии с частью 5 статьи 5 Федерального закона "Об обеспечении единства измерений" Правительство Российской Федерации постановляет:""",  
Отрывок документа №3
====================="""Header 1. Статья 

In [ ]:
#Вопрос пользователя
query_text = 'О чем Приказ №2906?'
gpt_answer = get_gpt_ya_response(ID_FOLDER, chunks_metadata_question(query_text))
print ("Ответ YaGPT:\n", "=================\n", gpt_answer)

0.45445522199798427
0.45661510924272264
0.465036980162414

Отрывок документа №1
====================="""Header 1. МИНИСТЕРСТВО ПРОМЫШЛЕННОСТИ И ТОРГОВЛИ РОССИЙСКОЙ ФЕДЕРАЦИИ ПРИКАЗ от 28 августа 2020 г. N 2907 ОБ УТВЕРЖДЕНИИ ПОРЯДКА УСТАНОВЛЕНИЯ И ИЗМЕНЕНИЯ ИНТЕРВАЛА МЕЖДУ ПОВЕРКАМИ СРЕДСТВ ИЗМЕРЕНИЙ, ПОРЯДКА УСТАНОВЛЕНИЯ, ОТМЕНЫ МЕТОДИК ПОВЕРКИ И ВНЕСЕНИЯ ИЗМЕНЕНИЙ В НИХ, ТРЕБОВАНИЙ К МЕТОДИКАМ ПОВЕРКИ СРЕДСТВ ИЗМЕРЕНИЙ,Header 2. пункт 2.""",, """2. Контроль за исполнением настоящего приказа возложить на заместителя Министра промышленности и торговли Российской Федерации А.С. Беспрозванных.
Врио Министра
Г.М.КАДЫРОВА
Приложение N 1
к приказу Минпромторга России
от 28 августа 2020 г. N 2907""",  
Отрывок документа №2
====================="""Header 1. МИНИСТЕРСТВО ПРОМЫШЛЕННОСТИ И ТОРГОВЛИ РОССИЙСКОЙ ФЕДЕРАЦИИ ПРИКАЗ от 28 августа 2020 г. N 2906 ОБ УТВЕРЖДЕНИИ ПОРЯДКА СОЗДАНИЯ И ВЕДЕНИЯ ФЕДЕРАЛЬНОГО ИНФОРМАЦИОННОГО ФОНДА ПО ОБЕСПЕЧЕНИЮ ЕДИНСТВА ИЗМЕРЕНИЙ, ПЕРЕДАЧИ СВЕДЕНИЙ В НЕГО И ВНЕС

In [ ]:
#Вопрос пользователя
query_text = 'ЦСМ отказал в поверке газового счетчика по причине отсутствия у меня действующего договора на техническое обслуживание внутридомового газового оборудования со специализированной организацией. Насколько правомерны действия ЦСМ.'
gpt_answer = get_gpt_ya_response(ID_FOLDER, chunks_metadata_question(query_text))
print ("Ответ YaGPT:\n", "=================\n", gpt_answer)

0.3889358579640556
0.40138068584939823
0.3945804720500916

Отрывок документа №1
====================="""Header 1. ТРЕБОВАНИЯ К СОДЕРЖАНИЮ СВИДЕТЕЛЬСТВА О ПОВЕРКЕ,Header 2. пункт 2.""",, """2. Настоящие требования применяются при оформлении результатов поверки средств измерений.""",  
Отрывок документа №2
====================="""Header 1. МИНИСТЕРСТВО ПРОМЫШЛЕННОСТИ И ТОРГОВЛИ РОССИЙСКОЙ ФЕДЕРАЦИИ ПРИКАЗ от 25 июня 2013 г. N 971 ОБ УТВЕРЖДЕНИИ АДМИНИСТРАТИВНОГО РЕГЛАМЕНТА ПО ПРЕДОСТАВЛЕНИЮ ФЕДЕРАЛЬНЫМ АГЕНТСТВОМ ПО ТЕХНИЧЕСКОМУ РЕГУЛИРОВАНИЮ И МЕТРОЛОГИИ ГОСУДАРСТВЕННОЙ УСЛУГИ ПО ОТНЕСЕНИЮ ТЕХНИЧЕСКИХ СРЕДСТВ К СРЕДСТВАМ ИЗМЕРЕНИЙ,Header 2. пункт 3.,Header 3. Исчерпывающий перечень оснований для приостановления или отказа в предоставлении государственной услуги,Header 4. пункт 19.""",, """19. Основания для отказа в предоставлении государственной услуги не предусмотрены законодательством Российской Федерации.
Если техническое средство не предназначено для выполнения количественной оценки

In [ ]:
#Вопрос пользователя
query_text = 'Что ты умеешь?'
gpt_answer = get_gpt_ya_response(ID_FOLDER, chunks_metadata_question(query_text))
print ("Ответ YaGPT:\n", "=================\n", gpt_answer)

0.19896301747632406
0.217582230591712
0.22344145300153562

Отрывок документа №1
====================="""Header 1. ПРАВИТЕЛЬСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ ПОСТАНОВЛЕНИЕ от 17 июня 2004 г. N 294 О ФЕДЕРАЛЬНОМ АГЕНТСТВЕ ПО ТЕХНИЧЕСКОМУ РЕГУЛИРОВАНИЮ И МЕТРОЛОГИИ,Header 2. II. Полномочия,Header 3. пункт 5.""",, """(пп. 5.11(2) введен Постановлением Правительства РФ от 13.05.2016 N 409)  
5.11(3). организует подготовку кадров и дополнительное профессиональное образование в сфере стандартизации;
(пп. 5.11(3) введен Постановлением Правительства РФ от 13.05.2016 N 409)  
5.11(4). обеспечивает научную и методическую поддержку проведения работ по стандартизации;
(пп. 5.11(4) введен Постановлением Правительства РФ от 13.05.2016 N 409)  
5.11(5). определяет порядок регистрации стандартов организаций, в том числе технических условий, в Федеральном информационном фонде стандартов;
(пп. 5.11(5) введен Постановлением Правительства РФ от 31.05.2021 N 834)  
5.11(6). определяет порядок разработки и утвержден

In [ ]:
#Вопрос пользователя
query_text = 'Привет!'
gpt_answer = get_gpt_ya_response(ID_FOLDER, chunks_metadata_question(query_text))
print ("Ответ YaGPT:\n", "=================\n", gpt_answer)

0.27152318136487574
0.29894436989114936
0.2731239574642409

Отрывок документа №1
====================="""Header 1. ПРАВИЛА ПО МЕЖГОСУДАРСТВЕННОЙ СТАНДАРТИЗАЦИИ ПОРЯДОК ПРИЗНАНИЯ РЕЗУЛЬТАТОВ ИСПЫТАНИЙ И УТВЕРЖДЕНИЯ ТИПА, ПЕРВИЧНОЙ ПОВЕРКИ, МЕТРОЛОГИЧЕСКОЙ АТТЕСТАЦИИ СРЕДСТВ ИЗМЕРЕНИЙ Procedure for recognition of test results and type approval, initial verification, metrological certification of measuring instruments ПМГ 06-2019,Header 2. 2 Основные положения,Header 3. пункт 2.""",, """Национальный орган в срок, не превышающий 30 календарных дней с даты получения заявки от национального органа государства - участника Соглашения, на территории которого заявитель осуществляет выпуск из производства СИ утвержденного типа, принимает решение о внесении изменений в методику поверки, и (или) изменении интервала между поверками, и (или) внесении изменений в сведения о поверочной лаборатории, проводящей первичную поверку, размещает сведения в информационном фонде в области обеспечения единства из

In [ ]:
#Вопрос пользователя
query_text = 'Почему идет дождь?'
gpt_answer = get_gpt_ya_response(ID_FOLDER, chunks_metadata_question(query_text))
print ("Ответ YaGPT:\n", "=================\n", gpt_answer)

0.18365930097595928
0.18424464957118913
0.18818066341695217

Отрывок документа №1
====================="""Header 1. РОССИЙСКАЯ ФЕДЕРАЦИЯ ФЕДЕРАЛЬНЫЙ ЗАКОН ОБ ОБЕСПЕЧЕНИИ ЕДИНСТВА ИЗМЕРЕНИЙ,Header 2. Статья 16. Утратила силу с 1 августа 2011 года. - Федеральный закон от 18.07.2011 N 242-ФЗ. Статья 17. Права и обязанности должностных лиц при осуществлении федерального государственного метрологического контроля (надзора)""",, """(в ред. Федеральных законов от 18.07.2011 N 242-ФЗ, от 11.06.2021 N 170-ФЗ)""",  
Отрывок документа №2
====================="""Header 1. ПОРЯДОК АТТЕСТАЦИИ ПЕРВИЧНЫХ РЕФЕРЕНТНЫХ МЕТОДИК (МЕТОДОВ) ИЗМЕРЕНИЙ, РЕФЕРЕНТНЫХ МЕТОДИК (МЕТОДОВ) ИЗМЕРЕНИЙ И МЕТОДИК (МЕТОДОВ) ИЗМЕРЕНИЙ И ИХ ПРИМЕНЕНИЯ""",, """I. Общие положения""",  
Отрывок документа №3
====================="""Header 1. ПРАВИТЕЛЬСТВО РОССИЙСКОЙ ФЕДЕРАЦИИ ПОСТАНОВЛЕНИЕ от 31 октября 2009 г. N 879 ОБ УТВЕРЖДЕНИИ ПОЛОЖЕНИЯ О ЕДИНИЦАХ ВЕЛИЧИН, ДОПУСКАЕМЫХ К ПРИМЕНЕНИЮ В РОССИЙСКОЙ ФЕДЕРАЦИИ,Header 2. ВНЕСИСТЕМ

In [ ]:
import pandas as pd

# Загрузка перечня вопросов-ответов из файла Excel в DataFrame
df = pd.read_excel('output.xlsx')
df.head()

,№ п/п,Вопрос,Идеальный ответ,Ответ GPT
0,1,Какие предъявляются требования к измерениям пр...,Законодательство Российской Федерации об обесп...,NaN
1,2,Какой нормативный правовой акт устанавливает п...,В соответствии с частью 1 статьи 9 Федеральног...,NaN
2,3,Допустимо ли проведение процедуры калибровки с...,Сфера государственного регулирования обеспечен...,NaN
3,4,Возможно ли использование средств измерений в ...,В связи с отсутствием в Вашем обращении информ...,NaN
4,5,"Нужно ли поверять все средства измерений, прим...","Управление метрологии, государственного контро...",NaN


In [ ]:
# Извлечение данных из столбца 'Вопрос' по очереди и запись ответа GPT в столбец Ответ YaGPT_chunks

for question in df['Вопрос']:
    df['Ответ YaGPT_chunks'] = df['Вопрос'].apply(lambda x: get_gpt_ya_response(ID_FOLDER, x))

In [ ]:
df

,№ п/п,Вопрос,Идеальный ответ,Ответ GPT,Ответ YaGPT_chunks
0,1,Какие предъявляются требования к измерениям пр...,Законодательство Российской Федерации об обесп...,NaN,В соответствии с Федеральным законом от 26.06....
1,2,Какой нормативный правовой акт устанавливает п...,В соответствии с частью 1 статьи 9 Федеральног...,NaN,В соответствии с Федеральным законом от 26.06....
2,3,Допустимо ли проведение процедуры калибровки с...,Сфера государственного регулирования обеспечен...,NaN,В соответствии с Федеральным законом «Об обесп...
3,4,Возможно ли использование средств измерений в ...,В связи с отсутствием в Вашем обращении информ...,NaN,В соответствии с Федеральным законом № 102-ФЗ ...
4,5,"Нужно ли поверять все средства измерений, прим...","Управление метрологии, государственного контро...",NaN,"**Нужно ли поверять все средства измерений, пр..."
5,6,"Как определить, относятся ли измерения, провод...",В соответствии со статьей 9 Федерального закон...,NaN,В соответствии с Федеральным законом № 102-ФЗ ...
6,7,Можно ли применять средство измерений неутверж...,Для применения средства измерений неутвержден...,NaN,В соответствии с Федеральным законом № 102-ФЗ ...
7,8,Просим рассмотреть предложения по доработке пр...,В соответствии с частью 4 статьи 13 Федерально...,NaN,В соответствии с Федеральным законом «Об обесп...
8,9,"Каковы особенности поверки средств измерений, ...",Согласно пункту 13 Порядка проведения поверки ...,NaN,"Особенности поверки средств измерений, поступи..."
9,10,Можете ли Вы предоставить выписку из Федеральн...,Предоставление из Федерального информационного...,NaN,Для получения выписки из Федерального информац...


In [ ]:
#Сохраним ответы обученной LLM в файл
df.to_excel('answer_YaGPT_bd_03_chunks+metadata.xlsx', index=False)